In [34]:
import re
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ####################################################################
# #################### CONFIGURATION — FILL THIS IN ##################
# ####################################################################
YEAR = 2024 # <-- CHANGE YEAR (e.g. 2019, 2024)
TICKER = "sbm"        # <-- CHANGE TICKER (e.g. "AD", "JU")
# ####################################################################

BASE = Path(".")
INPUT_FOLDER = BASE / "data_intermediate" / f"mdna_intermediate_{YEAR}"
CLEAN_FOLDER = BASE / "data_clean" / f"mdna_clean_{YEAR}"
MANIFEST_PATH = BASE / "outputs" / f"data_clean_manifest_{YEAR}.csv"

CLEAN_FOLDER.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

STOP_WORDS = set(ENGLISH_STOP_WORDS)

def technical_clean(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)
    text = re.sub(r"(?im)^\s*page\s+\d+\s*$", "", text)
    text = re.sub(r"(?m)([A-Za-z])-\n([A-Za-z])", r"\1\2", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = "\n".join([ln.rstrip() for ln in text.split("\n")])
    return text.strip()

def nlp_clean(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    tokens = [w for w in text.split() if w not in STOP_WORDS]
    return " ".join(tokens)

# Only process files for this ticker
files = sorted(INPUT_FOLDER.glob(f"{TICKER}_*.txt"))
if not files:
    raise FileNotFoundError(
        f"No files found for ticker '{TICKER}' in {INPUT_FOLDER.resolve()}"
    )

print(f"\nCleaning files for {TICKER}, YEAR={YEAR}:")
for f in files:
    print(" -", f.name)

records = []
for file in files:
    raw = file.read_text(encoding="utf-8", errors="ignore")
    final_clean = nlp_clean(technical_clean(raw))

    out_path = CLEAN_FOLDER / file.name
    out_path.write_text(final_clean, encoding="utf-8")

    records.append({
        "doc_id": file.stem,
        "year": YEAR,
        "ticker": TICKER,
        "clean_path": str(out_path),
        "n_words": len(final_clean.split())
    })

pd.DataFrame(records).to_csv(MANIFEST_PATH, index=False)

print(f"\nSaved {len(records)} clean files to: {CLEAN_FOLDER.resolve()}")
print(f"Saved manifest to: {MANIFEST_PATH.resolve()}")



Cleaning files for sbm, YEAR=2024:
 - sbm_2024.txt

Saved 1 clean files to: /Users/pnlmf036/Documents/Thesis_Textanalysis/data_clean/mdna_clean_2024
Saved manifest to: /Users/pnlmf036/Documents/Thesis_Textanalysis/outputs/data_clean_manifest_2024.csv
